# Stage 2 - Data Ingestestion

In [2]:
import config
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /Users/djaru/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
from config import (
    ARXIV_API_URL, CATEGORY, MAX_RESULTS, SLEEP_TIME,
    CHUNK_SIZE, CHUNK_OVERLAP, METADATA_PATH, CHUNKS_PATH, BASE_DIR
)

In [6]:
"""
Steps 1-2: Document Ingestion & Semantic Chunking
"""

import os
import json
import time
import logging
from typing import List, Dict
from urllib.parse import urlencode
import feedparser
import pandas as pd
from tenacity import retry, stop_after_attempt, wait_exponential
import nltk 
from nltk.tokenize import sent_tokenize

from config import (
    ARXIV_API_URL, CATEGORY, MAX_RESULTS, SLEEP_TIME,
    CHUNK_SIZE, CHUNK_OVERLAP, METADATA_PATH, CHUNKS_PATH
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Download NLTK data if not present
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')


@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=4, max=10))
def fetch_recent_arxiv_papers(max_results: int = MAX_RESULTS, start: int = 0) -> List[Dict]:
    """
    Fetch papers from ArXiv API with retry logic and error handling.
    """
    params = {
        "search_query": f"cat:{CATEGORY}",
        "start": start,
        "max_results": max_results,
        "sortBy": "submittedDate",
        "sortOrder": "descending",
    }
    query_url = f"{ARXIV_API_URL}?{urlencode(params)}"
    
    logger.info(f"Fetching papers from: {query_url}")
    response = feedparser.parse(query_url)
    
    # Error checking
    if response.bozo:
        logger.warning(f"Feed parsing warning: {response.bozo_exception}")
    
    papers = []
    for entry in response.entries:
        paper = {
            "doc_id": entry.id.split("/")[-1],
            "title": entry.title.replace("\n", " ").strip(),
            "authors": [author.name for author in entry.authors],
            "abstract": entry.summary.replace("\n", " ").strip(),
            "published": entry.published,
            "updated": entry.updated,
            "categories": [tag.term for tag in entry.tags],
            "pdf_url": next((link.href for link in entry.links 
                          if link.type == "application/pdf"), None),
            "arxiv_url": entry.id,
            "source": "arXiv"
        }
        papers.append(paper)
    
    time.sleep(SLEEP_TIME)
    logger.info(f"Retrieved {len(papers)} papers")
    return papers


def save_metadata_jsonl(papers: List[Dict], output_path: str):
    """Save papers to JSONL format."""
    with open(output_path, "w", encoding="utf-8") as f:
        for paper in papers:
            f.write(json.dumps(paper, ensure_ascii=False) + "\n")
    logger.info(f"Saved {len(papers)} papers to {output_path}")


def semantic_chunk_text(text: str, chunk_size: int = CHUNK_SIZE, 
                        overlap: int = CHUNK_OVERLAP) -> List[Dict]:
    """
    IMPROVED: Semantic chunking with sentence boundaries.
    """
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_length = 0
    chunk_idx = 0
    
    for sentence in sentences:
        sentence_length = len(sentence.split())
        
        # Start new chunk if current exceeds size
        if current_length + sentence_length > chunk_size and current_chunk:
            chunk_text = " ".join(current_chunk)
            chunks.append({
                "text": chunk_text,
                "token_count": current_length,
                "chunk_index": chunk_idx,
                "num_sentences": len(current_chunk)
            })
            chunk_idx += 1
            
            # Handle overlap by keeping last sentences
            overlap_words = 0
            overlap_sentences = []
            for s in reversed(current_chunk):
                s_len = len(s.split())
                if overlap_words + s_len <= overlap:
                    overlap_sentences.insert(0, s)
                    overlap_words += s_len
                else:
                    break
            
            current_chunk = overlap_sentences
            current_length = overlap_words
        
        current_chunk.append(sentence)
        current_length += sentence_length
    
    # Add final chunk
    if current_chunk:
        chunk_text = " ".join(current_chunk)
        chunks.append({
            "text": chunk_text,
            "token_count": current_length,
            "chunk_index": chunk_idx,
            "num_sentences": len(current_chunk)
        })
    
    return chunks


def process_papers_to_chunks(metadata_path: str = METADATA_PATH,
                             output_path: str = CHUNKS_PATH) -> pd.DataFrame:
    """
    Convert papers to chunks with metadata preservation.
    """
    logger.info("Loading papers and creating semantic chunks...")
    
    # Load papers
    papers = []
    with open(metadata_path, "r", encoding="utf-8") as f:
        for line in f:
            papers.append(json.loads(line))
    
    df = pd.DataFrame(papers)
    
    # Create chunks
    rag_chunks = []
    for _, row in df.iterrows():
        # Combine title and abstract for better context
        text = f"{row['title']}\n\n{row['abstract']}"
        
        chunks = semantic_chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)
        
        for chunk in chunks:
            rag_chunks.append({
                "chunk_id": f"{row['doc_id']}_chunk_{chunk['chunk_index']}",
                "doc_id": row["doc_id"],
                "chunk_index": chunk['chunk_index'],
                "text": chunk['text'],
                "token_count": chunk['token_count'],
                "num_sentences": chunk['num_sentences'],
                "title": row["title"],
                "authors": row["authors"],
                "categories": row["categories"],
                "published": row["published"],
                "arxiv_url": row["arxiv_url"],
                "source": row["source"]
            })
    
    # Save chunks
    chunks_df = pd.DataFrame(rag_chunks)
    with open(output_path, "w", encoding="utf-8") as f:
        for _, chunk in chunks_df.iterrows():
            f.write(json.dumps(chunk.to_dict(), ensure_ascii=False) + "\n")
    
    logger.info(f"Created {len(rag_chunks)} chunks from {len(df)} papers")
    return chunks_df


def run_ingestion():
    """Execute full ingestion pipeline."""
    logger.info("=" * 60)
    logger.info("STEP 1: Fetching Papers from ArXiv")
    logger.info("=" * 60)
    
    papers = fetch_recent_arxiv_papers(max_results=MAX_RESULTS)
    save_metadata_jsonl(papers, METADATA_PATH)
    
    logger.info("\n" + "=" * 60)
    logger.info("STEP 2: Creating Semantic Chunks")
    logger.info("=" * 60)
    
    chunks_df = process_papers_to_chunks()
    logger.info(f"\n✅ Ingestion complete! Files saved to {BASE_DIR}/")
    
    return chunks_df


if __name__ == "__main__":
    run_ingestion()

INFO:__main__:============================================================
INFO:__main__:STEP 1: Fetching Papers from ArXiv
INFO:__main__:============================================================
INFO:__main__:Fetching papers from: http://export.arxiv.org/api/query?search_query=cat%3Acs.AI&start=0&max_results=200&sortBy=submittedDate&sortOrder=descending
INFO:__main__:Retrieved 200 papers
INFO:__main__:Saved 200 papers to arxiv_cs_ai/papers_metadata.jsonl
INFO:__main__:
INFO:__main__:STEP 2: Creating Semantic Chunks
INFO:__main__:============================================================
INFO:__main__:Loading papers and creating semantic chunks...
INFO:__main__:Created 376 chunks from 200 papers
INFO:__main__:
✅ Ingestion complete! Files saved to arxiv_cs_ai/
